# 03 Feature Engineering
Generates price lags (1d, 3d, 7d, 14d), rolling medians, volatility metrics, external signals, and exports `featured_gpu_prices.csv`.

In [2]:
import sys
from pathlib import Path
import pandas as pd

workspace_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(workspace_dir) not in sys.path:
    sys.path.insert(0, str(workspace_dir))

from src.io_utils import load_sample_dataset, save_processed_dataset, load_yaml_config
from src.clean_prices import clean_price_observations
from src.normalize_prices import normalize_price_series
from src.feature_builder import build_gpu_features

## 1. Load Data & Configs

In [3]:
cfg = load_yaml_config("model_config.yaml")
products_df = load_sample_dataset("products.csv")
listings_df = load_sample_dataset("product_listings.csv")
obs_df = load_sample_dataset("daily_price_observations.csv")
ext_df = load_sample_dataset("external_market_signals.csv")

cleaned_obs = clean_price_observations(obs_df)
norm_df = normalize_price_series(cleaned_obs, listings_df, products_df)
print(f"Normalized dataset shape: {norm_df.shape}")

Normalized dataset shape: (169, 19)


## 2. Build Time-Series Lags & Rolling Features

In [4]:
lags = cfg["feature_engineering"]["lag_days"]
windows = cfg["feature_engineering"]["rolling_windows"]

featured_df = build_gpu_features(norm_df, external_df=ext_df, lag_days=lags, rolling_windows=windows)
display(featured_df[["date", "sku_id", "total_effective_price_inr", "price_lag_1d", "rolling_median_7d", "price_volatility_7d", "usd_inr_rate"]].head(10))

,date,sku_id,total_effective_price_inr,price_lag_1d,rolling_median_7d,price_volatility_7d,usd_inr_rate
0,2026-06-01,RTX5060TI-16G-ASUS-DUAL,64200,NaN,NaN,0.0000,83.50
1,2026-06-01,RTX5060TI-16G-ASUS-DUAL,64850,64200.0,64200.0,0.0000,83.50
2,2026-06-05,RTX5060TI-16G-ASUS-DUAL,66200,64850.0,64525.0,0.0071,83.55
3,2026-06-05,RTX5060TI-16G-ASUS-DUAL,66850,66200.0,64850.0,0.0157,83.55
4,2026-06-10,RTX5060TI-16G-ASUS-DUAL,68800,66850.0,65525.0,0.0185,83.62
5,2026-06-10,RTX5060TI-16G-ASUS-DUAL,69450,68800.0,66200.0,0.0272,83.62
6,2026-06-15,RTX5060TI-16G-ASUS-DUAL,71800,69450.0,66525.0,0.0315,83.70
7,2026-06-15,RTX5060TI-16G-ASUS-DUAL,72450,71800.0,66850.0,0.0405,83.70
8,2026-06-20,RTX5060TI-16G-ASUS-DUAL,74900,72450.0,68800.0,0.0414,83.78
9,2026-06-20,RTX5060TI-16G-ASUS-DUAL,75550,74900.0,69450.0,0.0453,83.78


## 3. Save Processed Feature Dataset

In [5]:
out_path = save_processed_dataset(featured_df, "featured_gpu_prices.csv")
print(f"Successfully exported featured GPU price dataset to: {out_path}")

Successfully exported featured GPU price dataset to: /workspace/notebooks/gpu_price_forecasting/data/processed/featured_gpu_prices.csv
